# Phase 2a — Teacher Generates Distillation Targets with vLLM

**Goal:** Use `Qwen2.5-7B-Instruct` to generate summaries on 10k CNN/DailyMail training examples. These teacher-generated summaries become the training targets for Phase 2b (sequence-level KD).

**Output:** `teacher_generations.jsonl` — one record per example with `{idx, article, gold_summary, teacher_summary}`.

**Hardware target:** Single large-memory A100-class GPU. This version is tuned for ~100GB VRAM and uses vLLM for faster offline inference.

**Why vLLM:** vLLM improves throughput through continuous batching and efficient KV-cache memory management, which is a better fit for large offline generation jobs than plain `transformers.generate()`.


In [ ]:
# vLLM installs/uses compatible inference dependencies.
# If your cluster already has vLLM installed, you can skip this cell.
!pip install -q -U vllm datasets==2.21.0 sentencepiece


In [ ]:
import gc, json, re, time
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB')
else:
    print('GPU: none')

CONFIG = {
    'teacher_model': 'Qwen/Qwen2.5-7B-Instruct',
    'dataset': 'cnn_dailymail',
    'dataset_config': '3.0.0',
    'train_split': 'train',
    'num_train_samples': 10000,

    # A100 ~100GB recommendation for Qwen2.5-7B-Instruct with ~3k prompt tokens + 160 output tokens.
    # Start here. If stable, try 96 or 128. If OOM, reduce to 32.
    'batch_size': 64,

    'max_input_tokens': 3000,
    'max_new_tokens': 160,

    # vLLM engine controls. Keep max_num_seqs aligned with batch_size for predictable memory use.
    'vllm_gpu_memory_utilization': 0.92,
    'vllm_max_num_seqs': 64,
    'vllm_max_num_batched_tokens': 32768,

    'seed': 42,
    'output_dir': './kd_teacher_data',
    'output_file': 'teacher_generations.jsonl',
    'save_every_n_batches': 5,
}
Path(CONFIG['output_dir']).mkdir(exist_ok=True)
torch.manual_seed(CONFIG['seed'])
print(json.dumps(CONFIG, indent=2))


In [ ]:
# Load training split, shuffle, take subset
ds = load_dataset(CONFIG['dataset'], CONFIG['dataset_config'], split=CONFIG['train_split'])
ds = ds.shuffle(seed=CONFIG['seed']).select(range(CONFIG['num_train_samples']))
print(f'Loaded {len(ds)} training examples')
print('Sample article:', ds[0]['article'][:200], '...')
print('Sample gold:', ds[0]['highlights'])

In [ ]:
SYSTEM_PROMPT = (
    'You are a concise news summarizer. Write a short summary of the article in 2-3 sentences. '
    'Output only the summary itself, with no preamble, headers, or commentary.'
)
USER_TEMPLATE = 'Article:\n{article}\n\nSummary:'

PREAMBLE_RE = re.compile(
    r'^\s*(here(?:\s+is|\'s)?\s+(?:a\s+)?(?:brief\s+|short\s+|concise\s+)?summary[:\s\-]*|'
    r'summary[:\s\-]+)',
    re.IGNORECASE,
)

def clean_pred(text: str) -> str:
    return PREAMBLE_RE.sub('', text.strip()).strip().lstrip('\n').strip()

def build_prompt(article: str) -> str:
    """Build a Qwen chat prompt while preserving the final generation marker.

    Avoid tokenizer(..., truncation=True) on the full prompt because that can truncate
    the end of the chat template. Instead, truncate only the article content.
    """
    empty_msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_TEMPLATE.format(article='')},
    ]
    overhead_ids = tokenizer.apply_chat_template(
        empty_msgs, tokenize=True, add_generation_prompt=True
    )
    max_article_tokens = max(256, CONFIG['max_input_tokens'] - len(overhead_ids) - 8)

    article_ids = tokenizer(
        article, add_special_tokens=False, truncation=True, max_length=max_article_tokens
    ).input_ids
    trimmed_article = tokenizer.decode(article_ids, skip_special_tokens=True)

    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_TEMPLATE.format(article=trimmed_article)},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


In [ ]:
# Load tokenizer + vLLM teacher engine
tokenizer = AutoTokenizer.from_pretrained(CONFIG['teacher_model'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=CONFIG['max_new_tokens'],
    skip_special_tokens=True,
)

llm = LLM(
    model=CONFIG['teacher_model'],
    dtype='bfloat16',
    tensor_parallel_size=1,
    trust_remote_code=True,
    gpu_memory_utilization=CONFIG['vllm_gpu_memory_utilization'],
    max_model_len=CONFIG['max_input_tokens'] + CONFIG['max_new_tokens'],
    max_num_seqs=CONFIG['vllm_max_num_seqs'],
    max_num_batched_tokens=CONFIG['vllm_max_num_batched_tokens'],
    enable_prefix_caching=True,
)
print('vLLM teacher loaded.')


In [ ]:
# Resume support: if output file exists, skip already-generated indices
out_path = Path(CONFIG['output_dir']) / CONFIG['output_file']
completed_indices = set()
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            try:
                completed_indices.add(json.loads(line)['idx'])
            except Exception:
                pass
    print(f'Resume: {len(completed_indices)} examples already done, skipping those.')

to_process = [i for i in range(len(ds)) if i not in completed_indices]
print(f'To process: {len(to_process)}')

In [ ]:
bs = CONFIG['batch_size']
start = time.time()
buffer = []  # records waiting to be flushed
total = len(to_process)

for batch_start in range(0, total, bs):
    batch_idx = to_process[batch_start:batch_start + bs]
    batch_articles = [ds[i]['article'] for i in batch_idx]
    batch_golds = [ds[i]['highlights'] for i in batch_idx]

    prompts = [build_prompt(art) for art in batch_articles]

    # vLLM performs efficient scheduling/continuous batching internally.
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    decoded = [out.outputs[0].text for out in outputs]

    for idx, art, gold, raw in zip(batch_idx, batch_articles, batch_golds, decoded):
        buffer.append({
            'idx': idx,
            'article': art,
            'gold_summary': gold,
            'teacher_summary': clean_pred(raw),
            'teacher_summary_raw': raw.strip(),
        })

    processed_batches = (batch_start // bs) + 1
    done = batch_start + len(batch_idx)

    # Flush periodically. JSONL append makes resume easy after interruption.
    if processed_batches % CONFIG['save_every_n_batches'] == 0 and buffer:
        with open(out_path, 'a', encoding='utf-8') as f:
            for rec in buffer:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        buffer = []

        elapsed = time.time() - start
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        print(f'  [{done}/{total}] elapsed={elapsed:.0f}s rate={rate:.2f}/s eta_min={eta/60:.1f}')

# Final flush
if buffer:
    with open(out_path, 'a', encoding='utf-8') as f:
        for rec in buffer:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'\nDone in {(time.time() - start) / 60:.1f} min. Output: {out_path}')


In [ ]:
# Sanity check on the output file
with open(out_path) as f:
    lines = f.readlines()
print(f'Total records: {len(lines)}')
print('\nFirst 3 teacher generations:')
for line in lines[:3]:
    rec = json.loads(line)
    print(f"\n[idx={rec['idx']}]")
    print(f"  gold:    {rec['gold_summary']}")
    print(f"  teacher: {rec['teacher_summary']}")

In [ ]:
# Free VRAM before moving to the next notebook
del llm, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Teacher unloaded. Ready for Phase 2b.')
